In [ ]:
import pysam
import numpy as np
import pandas as pd
import pyBigWig

In [ ]:
def wig_to_bigwig(wig_path, chrom_sizes_path, bw_path):
    chroms = {}
    with open(chrom_sizes_path) as f:
        for line in f:
            name, size = line.strip().split("\t")
            chroms[name] = int(size)

    entries = {} 
    chrom, start, step, span = None, None, None, None

    with open(wig_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("fixedStep"):
                parts = dict(p.split("=") for p in line.split()[1:])
                chrom = parts["chrom"]
                start = int(parts["start"]) - 1
                step = int(parts.get("step", 1))
                span = int(parts.get("span", 1))
                if chrom not in entries:
                    entries[chrom] = []
            elif line.startswith("variableStep"):
                parts = dict(p.split("=") for p in line.split()[1:])
                chrom = parts["chrom"]
                span = int(parts.get("span", 1))
                start = None
                if chrom not in entries:
                    entries[chrom] = []
            elif line and not line.startswith("track"):
                if start is not None:  # fixedStep
                    val = float(line)
                    entries[chrom].append((start, start + span, val))
                    start += step
                else:  # variableStep
                    pos, val = line.split()
                    pos = int(pos) - 1
                    entries[chrom].append((pos, pos + span, float(val)))

    bw = pyBigWig.open(bw_path, "w")
    bw.addHeader(list(chroms.items()))

    for chrom in chroms:
        if chrom not in entries or not entries[chrom]:
            continue
        e = sorted(entries[chrom], key=lambda x: x[0])
        starts = [x[0] for x in e]
        ends = [x[1] for x in e]
        values = [x[2] for x in e]
        bw.addEntries([chrom] * len(e), starts, ends=ends, values=values)

    bw.close()

#wig_to_bigwig("/mnt/unallab/jleslie/20250827_ribosome_profiling_snf5_cyc8_ade2/28_29merAnalysis/45862_mono_S37_L004_R1_001_Aligned_unique_sorted_fwd.wig", "/mnt/unallab/jleslie/W303star_updated/chrNameLength.txt", "45862_mono_S37_L004_R1_001_Aligned_unique_sorted_fwd.bw")

In [ ]:
for sample in ["44143_mono_S30_L004_R1_001_Aligned_unique_sorted_fwd", "44143_mono_S30_L004_R1_001_Aligned_unique_sorted_rev"]:
        wig_to_bigwig(f"/mnt/unallab/jleslie/20250827_ribosome_profiling_snf5_cyc8_ade2/28_29merAnalysis/{sample}.wig", 
                        "/mnt/unallab/jleslie/W303star_updated/chrNameLength.txt",
                       f"/mnt/unallab/jleslie/20250827_ribosome_profiling_snf5_cyc8_ade2/28_29merAnalysis/{sample}.bw")

In [ ]:
import pyBigWig
import numpy as np
import matplotlib.pyplot as plt

def metagene_from_bigwig(bw_path, bed_path, strand_filter, window=100):
    bw = pyBigWig.open(bw_path)
    profile = np.zeros(window)
    with open(bed_path) as f:
        for line in f:
            chrom, start, end, name, score, strand = line.strip().split("\t")[:6]
            start, end = int(start), int(end)
            if strand != strand_filter:
                continue
            if strand == "+":
                anchor = end - 100
                region_start = anchor - 50
                region_end = anchor + 50
            else:
                anchor = start + 100
                region_start = anchor - 50
                region_end = anchor + 50
            # Bounds check
            chrom_len = bw.chroms().get(chrom, 0)
            if region_start < 0 or region_end > chrom_len:
                continue
            if region_end - region_start != window:
                continue
            vals = bw.values(chrom, region_start, region_end)
            vals = np.nan_to_num(vals)
            if strand == "-":
                vals = vals[::-1]
            profile += vals
    bw.close()
    return profile

bed_path = "/mnt/unallab/jleslie/w303genome/W303_minimal_CDS_output_addcolumn_100upstream-downstream.bed"
bw_dir = "/mnt/unallab/jleslie/20250827_ribosome_profiling_snf5_cyc8_ade2/28_29merAnalysis"

fwd_profile = metagene_from_bigwig(f"{bw_dir}/45862_mono_S37_L004_R1_001_Aligned_unique_sorted_fwd.bw", bed_path, "+")
rev_profile = metagene_from_bigwig(f"{bw_dir}/45862_mono_S37_L004_R1_001_Aligned_unique_sorted_rev.bw", bed_path, "-")
total_profile = fwd_profile + rev_profile

np.savetxt("45862_metagene.csv", total_profile, delimiter=",")

x = np.arange(100) - 50
plt.figure(figsize=(10, 4))
plt.plot(x, total_profile)
plt.axvline(0, color='red', linestyle='--', label='Stop codon')
for i in range(x[0], x[-1] + 1, 3):
    plt.axvline(i, color='black', linestyle=':', linewidth=0.5, alpha=0.5)
plt.xlabel('Position relative to stop codon (nt)')
plt.ylabel('Summed A-site counts')
plt.title('Metagene profile — 50 nt CDS + 50 nt downstream')
plt.legend()
plt.tight_layout()
plt.show()